### import libraries

In [2]:
import concurrent.futures
import requests
import concurrent.futures
import pandas as pd
import time


### define 2 functions that work together to get the weather data from the API

In [6]:
# Function that gets data for a whole year and caluculates the monthly average of a given api-parameter

def get_them_api(city, lat, lon, api_parameter):

    """
    Gets average monthly temperature for a City.
    Args:
        city (str): City name.
        lat (float): Latitude of the city.
        lon (float): Longitude of the city.
        api_parameter (str): API parameter for weather data.
    Returns:
        None
    """

    BASE_URL = "https://archive-api.open-meteo.com/v1/era5"
    #adds the weather data
    monthly_averages = []  # Renamed variable to store monthly averages
    # Loop through the months
    for month in range(1, 13):
        month_start = f"2020-{month:02}-01"
        # Get the last day of the month
        month_end = f"2020-{month:02}-31"
        # Loop for month with 30 days
        if month in [4, 6, 9, 11]:
            month_end = f"2020-{month:02}-30"
        # Loop for February
        elif month == 2:
            month_end = f"2020-{month:02}-28"  #
        # format the API URL
        url = f"{BASE_URL}?latitude={lat}&longitude={lon}&start_date={month_start}&end_date={month_end}&hourly={api_parameter}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            temps = data['hourly'][f"{api_parameter}"]
            # calculate the monthly average temperature
            if len(temps) > 0:
                monthly_avg = round(sum(temps) / len(temps), 2)
            else:
                monthly_avg = None
            monthly_averages.append(monthly_avg)  # Append to the list

    # add the monthly average temperatures to the data list
    monthly = {
        'city': city,
        'jan': monthly_averages[0],
        'feb': monthly_averages[1],
        'mar': monthly_averages[2],
        'apr': monthly_averages[3],
        'may': monthly_averages[4],
        'jun': monthly_averages[5],
        'jul': monthly_averages[6],
        'aug': monthly_averages[7],
        'sep': monthly_averages[8],
        'oct': monthly_averages[9],
        'nov': monthly_averages[10],
        'dec': monthly_averages[11],
    }
    return monthly


In [5]:
# Function that uses the "get_them_api" function to fetch weather data for cities in multiple CSV files using concurrent.futures.

def go_getter(csv_files_list, api_parameter, merged_filename):
    """
    Fetches weather data for cities in multiple CSV files using concurrent.futures.

    Args:
        csv_files_list (list): List of CSV file paths.
        api_parameter (str): API parameter for weather data.
        merged_filename (str): Filename for the merged data.

    Returns:
        None
    """
    # Loop through the CSV files (countries) and fetch weather data for each city in country
    for csv_file in csv_files_list:
        # Read the CSV file
        df = pd.read_csv(csv_file)
        # Fetch weather data for each city using concurrent.futures for faster processing
        with concurrent.futures.ThreadPoolExecutor() as executor:
            # Add a delay of 25 seconds before making the API request
            time.sleep(25)
            results = [executor.submit(get_them_api, row['name'], row['latitude'], row['longitude'], api_parameter) for _, row in df.iterrows()]
            concurrent.futures.wait(results)
            data_list = [result.result() for result in results]
        # Transform data_list to a DataFrame
        df_weather = pd.DataFrame(data_list)
        # Merge the weather data with the original DataFrame
        df = pd.concat([df, df_weather], axis=1)
        # Modify the filename
        naming = csv_file.replace("Cities.csv", f"{merged_filename}.csv")
        # Save the merged data to the CSV file with modified filename
        df.to_csv(naming, index=False)


## ALL PARAMETERS that can be used for the go_getter function and the &hourly= parameter
#### see [Open-Meteo Documentation](https://open-meteo.com/en/docs) for more information


temperature_2m_max
temperature_2m_min	°C (°F)	Maximum and minimum daily air temperature at 2 meters above ground
apparent_temperature_max
apparent_temperature_min	°C (°F)	Maximum and minimum daily apparent temperature
precipitation_sum	mm	Sum of daily precipitation (including rain, showers and snowfall)
rain_sum	mm	Sum of daily rain
showers_sum	mm	Sum of daily showers
snowfall_sum	cm	Sum of daily snowfall
precipitation_hours	hours	The number of hours with rain
precipitation_probability_max
precipitation_probability_min
precipitation_probability_mean	%	Probability of precipitation
weathercode	WMO code	The most severe weather condition on a given day
sunrise
sunset	iso8601	Sun rise and set times
windspeed_10m_max
windgusts_10m_max	km/h (mph, m/s, knots)	Maximum wind speed and gusts on a day
winddirection_10m_dominant	°	Dominant wind direction
shortwave_radiation_sum	MJ/m²	The sum of solar radiation on a given day in Megajoules
et0_fao_evapotranspiration	mm	Daily sum of ET₀ Reference Evapotranspiration of a well watered grass field
uv_index_max
uv_index_clear_sky_max	Index	Daily maximum in UV Index starting from 0. uv_index_clear_sky_max assumes cloud free conditions. Please follow the official WMO guidelines for ultraviolet index.

### European Country Codes (ISO 3166-1 alpha-2) for file names replication

In [3]:
european_country_codes = ['AL', 'AD', 'AM', 'AT', 'AZ', 'BY', 'BE', 'BA', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FO', 'FI', 'FR', 'GE', 'DE', 'GI', 'GR', 'GL', 'GG', 'HU', 'IS', 'IE', 'IM', 'IT', 'JE', 'KZ', 'XK', 'LV', 'LT', 'LU', 'MK', 'MT', 'MD', 'MC', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'RU', 'RS', 'SK', 'SI', 'ES', 'SE', 'CH', 'UA', 'GB', 'TR', 'UZ']


#clone CSV_files names from folder to a list (code = country_code)
csv_files = [code + "_Cities.csv" for code in european_country_codes]


### Each run with the go_getter function takes roughly 40-45 mins (70 mio api Calls) the sleep timer is set to 25 seconds to avoid api rate limit which also increases wait time

The go_getter function needs to run each time for every year/paramter combination to get the data

In [ ]:
# check if the csv_files contain the right file before running the go_getter function to fetch weather data
print(csv_files)

In [7]:
# check if the csv_files contain the right number of files
csv_files
# run the go_getter function for the temperature_2m parameter "temp_21" is the filename which will be added to the csv_files name
go_getter(csv_files, "temperature_2m", "Temp_20")

# After running the function merge all the created files in to one csv

### clone the files from the go_getter function and add to a list

In [19]:
csv_files_temp = [code + "_CitiesTemp_20.csv" for code in european_country_codes]

csv_files_temp

['AL_CitiesTemp_20.csv',
 'AD_CitiesTemp_20.csv',
 'AM_CitiesTemp_20.csv',
 'AT_CitiesTemp_20.csv',
 'AZ_CitiesTemp_20.csv',
 'BY_CitiesTemp_20.csv',
 'BE_CitiesTemp_20.csv',
 'BA_CitiesTemp_20.csv',
 'BG_CitiesTemp_20.csv',
 'HR_CitiesTemp_20.csv',
 'CY_CitiesTemp_20.csv',
 'CZ_CitiesTemp_20.csv',
 'DK_CitiesTemp_20.csv',
 'EE_CitiesTemp_20.csv',
 'FO_CitiesTemp_20.csv',
 'FI_CitiesTemp_20.csv',
 'FR_CitiesTemp_20.csv',
 'GE_CitiesTemp_20.csv',
 'DE_CitiesTemp_20.csv',
 'GI_CitiesTemp_20.csv',
 'GR_CitiesTemp_20.csv',
 'GL_CitiesTemp_20.csv',
 'GG_CitiesTemp_20.csv',
 'HU_CitiesTemp_20.csv',
 'IS_CitiesTemp_20.csv',
 'IE_CitiesTemp_20.csv',
 'IM_CitiesTemp_20.csv',
 'IT_CitiesTemp_20.csv',
 'JE_CitiesTemp_20.csv',
 'KZ_CitiesTemp_20.csv',
 'XK_CitiesTemp_20.csv',
 'LV_CitiesTemp_20.csv',
 'LT_CitiesTemp_20.csv',
 'LU_CitiesTemp_20.csv',
 'MK_CitiesTemp_20.csv',
 'MT_CitiesTemp_20.csv',
 'MD_CitiesTemp_20.csv',
 'MC_CitiesTemp_20.csv',
 'ME_CitiesTemp_20.csv',
 'NL_CitiesTemp_20.csv',


In [20]:
all_temp = pd.concat([pd.read_csv(f) for f in csv_files_temp])


#remove column "names"
all_temp = all_temp.drop(columns=['name'])

#remove duplicates
all_temp = all_temp.drop_duplicates()

#to csv
all_temp.to_csv('TEMP_20.csv', index=False)

In [21]:
all_temp

,latitude,longitude,country,population,id,city,jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec
0,39.87534,20.00477,AL,15147,363243,Sarandë,10.71,11.96,13.25,15.17,19.70,22.43,26.40,27.06,24.92,19.90,15.99,13.70
1,40.90250,20.65250,AL,61530,781988,Pogradec,3.42,4.85,7.52,10.19,16.03,17.77,23.20,23.74,20.81,14.12,8.59,6.37
2,42.07694,20.42194,AL,17832,782661,Kukës,3.65,5.80,7.78,11.31,16.61,19.20,24.07,24.06,21.10,14.32,9.05,6.51
3,40.61861,20.78083,AL,58259,782756,Korçë,2.57,3.96,6.84,9.19,15.65,17.40,22.94,23.36,20.03,13.25,7.70,5.39
4,40.07583,20.13889,AL,23437,783148,Gjirokastër,7.64,9.09,10.94,13.11,18.17,20.16,25.22,25.22,22.75,17.20,13.03,11.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,40.78206,72.34424,UZ,447800,1514588,Andijon,0.66,4.89,11.53,16.55,23.30,27.14,29.76,28.19,21.90,13.61,4.46,0.24
99,40.39194,71.47417,UZ,25543,1514615,Oltiariq,2.13,6.67,12.41,17.24,24.06,27.69,29.80,28.31,22.19,13.92,5.79,1.73
100,40.81955,72.74234,UZ,32750,1514716,Oyim,0.42,3.94,10.72,15.21,20.99,25.12,28.00,26.51,20.60,12.67,3.82,0.25
101,40.52204,72.07292,UZ,33167,1526979,Quva,1.04,5.50,12.14,17.16,23.86,27.52,29.89,28.34,22.44,14.07,5.04,0.69


##